In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from rag_evaluator import RAGEvaluator
from typing_extensions import Annotated,TypedDict
from langchain_ollama import ChatOllama
import json
import numpy as np

<h5>Define Retrieval Scoring Methodology </h5>
<a href =https://docs.smith.langchain.com/evaluation/tutorials/rag>Reference for code

In [ ]:
#define a structured output for the LLM to provide feedback in
class RetrievalRelevanceGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain why you gave the relevance score you did"]
    relevance_score: Annotated[int, ..., "Overall rating of how relevant the documents are to the question"]

#Prompt to feed into LLM to score the context
retrieval_relevance_instructions = """You are a teach grading how well the provided facts are related to the question. 

Using the following QUESTION: {question} and a set of FACTS: {context} provided by the student. 

Here is the grade criteria to follow:
(1) You goal is to identify FACTS that are completely unrelated to the QUESTION
(2) If the facts contain ANY keywords or semantic meaning related to the question, consider them relevant
(3) It is OK if the facts have SOME information that is unrelated to the question as long as (2) is met

Relevance:
A relevance score of 10 means that the FACTS contain ALL keywords or semantic meaning related to the QUESTION and are therefore relevant.
A relevance score of 5 means that the FACTS contain some keywords or semantic meaning related to the QUESTION but do not reference the specific entity.
A relevance score of 1 means that the FACTS are completely unrelated to the QUESTION.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset.
Give a relevance score on a scale of 1-10"""
retrieval_prompt = ChatPromptTemplate.from_template(retrieval_relevance_instructions)

#Define the llm to grade the relevance score  using the prompt and return in the structured ouptu
retrieval_relevance_llm = ChatOllama(model="llama3.1").with_structured_output(
    RetrievalRelevanceGrade,method="json_schema"
)
def retrieval_relevance(query, documents):
    '''
    Return the LLM's given relevance score of the documents to the query and
    provide and explanation for the score.
    '''
    doc_string = "\n\n".join(doc['text'] for doc in documents)

    #Build prompt based on query and context
    messages = retrieval_prompt.invoke({"question": query, "context": doc_string})
    #get response from grader llm
    response = retrieval_relevance_llm.invoke(messages)
    return response["relevance_score"], response["explanation"]

In [ ]:
def evaluate_retriever(rag_comp_dict):
    '''
    Caculate relevance score and retrieved ticker accuracy on queries and
    retrieved context stored in the saved RAG response dictionary
    '''
    results ={}
    query_dict = {}
    query_scores = []
    relevant_ticker_scores = []

    for query in rag_comp_dict["results"]:
        #calculate relevance_score for each question
        relevance_score, explanation = retrieval_relevance(query["query"], query["retrieved_context"])
        query_scores.append(relevance_score)

        #calculate accuracy of document tickers for each retrieved context
        doc_count = 0
        doc_ticker_relevant = 0
        expected_ticker = query["expected_ticker"]
        for doc in query["retrieved_context"]:
            doc_count +=1
            try:
                if len(doc["metadata"]) != 0:
                    if doc["metadata"]["ticker"] in expected_ticker:
                        doc_ticker_relevant +=1
            except: 
                if len(doc["metadata"]) != 0:
                    if doc["metadata"]["ticker"] == expected_ticker:
                        doc_ticker_relevant +=1

        if(doc_count == 0):
            ticker_relevance_accuracy =0
        else:
            ticker_relevance_accuracy = doc_ticker_relevant/doc_count
        relevant_ticker_scores.append(ticker_relevance_accuracy)

        query_dict[query["query"]] = {"response":query["response"],"context":query["retrieved_context"],
                                      "relevance_score": relevance_score, "relevance_explanation":explanation,"ticker_relevance_accuracy":ticker_relevance_accuracy}
        
    #calculate average relevance score/ticker accuracy accross questions
    avg_relevance_score = sum(query_scores)/len(query_scores)
    relevant_ticker_accuracy = np.mean(np.array(relevant_ticker_scores))

    results["query_results_retriever"] = query_dict
    results["avg_relevance_score"] = avg_relevance_score
    results["relevant_ticker_accuracy"] = relevant_ticker_accuracy
    return results

<h5> Define evaluator </h5>

<a href = "https://github.com/AIAnytime/rag-evaluator">Reference for Code

In [ ]:
def evaluate_generator(rag_comp_dict):
    '''
    Caculate generator metrics on queries and responses stored
    in the saved RAG response dictionary
    '''
    #Use RAGEvaluator package to calculate generator metrics based on query and ground truth answer
    results ={}
    evaluator = RAGEvaluator()
    query_dict = {}
    query_scores = []
    for query in rag_comp_dict["results"]:
        metrics = evaluator.evaluate_all(query["query"], query["response"], query["gt_answer"])
        query_scores.append(metrics)
        query_dict[query["query"]] = [query["response"], query["gt_answer"], metrics]

    #Get average score for each metric
    sums ={}
    counts = {} 
    for query in query_scores:
        for metric, value in query.items():
            # Sum up the values
            sums[metric] = sums.get(metric, 0) + value
            # Count the occurrences of each key
            counts[metric] = counts.get(metric, 0) + 1
    average_metrics = {key: sums[key] / counts[key] for key in sums}
    results["query_results_generator"] = query_dict
    results["average_metrics_generator"] = average_metrics
    return results


In [ ]:
def gen_rag_evaluation_dict(rag_comp_dict):
    '''
    Combine generator, retrieval, and timing scores into a single dictionary for the RAG
    '''
    results = {}
    query_times = []
    queries = []
    query_time_dict = {}
    query_time_results = {}
    for query in rag_comp_dict["results"]:
        time = query["query_time"]
        query_times.append(time)
        queries.append(query["query"])
        query_time_dict[query["query"]] = query["query_time"]
    avg_query_time = sum(query_times)/len(query_times)
    query_time_results["avg_query_time"] = avg_query_time
    query_time_results["query_times"] = query_time_dict

    results["generator_scores"] = evaluate_generator(rag_comp_dict)
    results["retrieval_scores"] = evaluate_retriever(rag_comp_dict)
    results["query_time_results"] = query_time_results
    return results
    


RAG Model 1

In [ ]:
with open('./response_dicts/rag_responses_model_1.json', 'r') as json_file:
    rag_model_1 = json.load(json_file)

In [ ]:
rag_scores_model_1 = gen_rag_evaluation_dict(rag_model_1)
with open('./score_dicts/rag_scores_model_1.json', 'w') as json_file:
    json.dump(rag_scores_model_1,json_file)

RAG Model 2

In [ ]:
with open('./response_dicts/rag_responses_model_2.json', 'r') as json_file:
    rag_model_2 = json.load(json_file)
rag_scores_model_2 = gen_rag_evaluation_dict(rag_model_2)
with open('./score_dicts/rag_scores_model_2.json', 'w') as json_file:
    json.dump(rag_scores_model_2,json_file)

RAG Model 3

In [ ]:
with open('./response_dicts/rag_responses_model_3.json', 'r') as json_file:
    rag_model_3 = json.load(json_file)

rag_scores_model_3 = gen_rag_evaluation_dict(rag_model_3)
with open('./score_dicts/rag_scores_model_3.json', 'w') as json_file:
    json.dump(rag_scores_model_3,json_file)

RAG Model 4

In [ ]:
with open('./response_dicts/rag_responses_model_2.json', 'r') as json_file:
    rag_model_4 = json.load(json_file)
rag_scores_model_4 = gen_rag_evaluation_dict(rag_model_4)
with open('./score_dicts/rag_scores_model_4.json', 'w') as json_file:
    json.dump(rag_scores_model_4,json_file)

RAG Model 5

In [ ]:
with open('./response_dicts/rag_responses_model_5.json', 'r') as json_file:
    rag_model_5 = json.load(json_file)
rag_scores_model_5 = gen_rag_evaluation_dict(rag_model_5)
with open('./score_dicts/rag_scores_model_5.json', 'w') as json_file:
    json.dump(rag_scores_model_5,json_file)

RAG Model 6

In [ ]:
with open('./response_dicts/rag_responses_model_6.json', 'r') as json_file:
    rag_model_6 = json.load(json_file)
rag_scores_model_6 = gen_rag_evaluation_dict(rag_model_6)
with open('./score_dicts/rag_scores_model_6.json', 'w') as json_file:
    json.dump(rag_scores_model_6,json_file)

RAG Model 7

In [ ]:
with open('./response_dicts/rag_responses_model_7.json', 'r') as json_file:
    rag_model_7 = json.load(json_file)
rag_scores_model_7 = gen_rag_evaluation_dict(rag_model_7)
with open('./score_dicts/rag_scores_model_7.json', 'w') as json_file:
    json.dump(rag_scores_model_7,json_file)

RAG Model 8

In [ ]:
with open('./response_dicts/rag_responses_model_8.json', 'r') as json_file:
    rag_model_8 = json.load(json_file)
rag_scores_model_8 = gen_rag_evaluation_dict(rag_model_8)
with open('./score_dicts/rag_scores_model_8.json', 'w') as json_file:
    json.dump(rag_scores_model_8,json_file)

RAG Model 9

In [ ]:
with open('./response_dicts/rag_responses_model_9.json', 'r') as json_file:
    rag_model_9 = json.load(json_file)
rag_scores_model_9 = gen_rag_evaluation_dict(rag_model_9)
with open('./score_dicts/rag_scores_model_9.json', 'w') as json_file:
    json.dump(rag_scores_model_9,json_file)

RAG Model 10

In [ ]:
with open('./response_dicts/rag_responses_model_10.json', 'r') as json_file:
    rag_model_10 = json.load(json_file)
rag_scores_model_10 = gen_rag_evaluation_dict(rag_model_10)
with open('./score_dicts/rag_scores_model_10.json', 'w') as json_file:
    json.dump(rag_scores_model_10,json_file)